In [18]:
from pathlib import Path

data_path = Path(
    r"C:\Users\Lenovo\Desktop\Customer_Segmentation_RFM\data\Online Retail.xlsx"
)

print("File exists:", data_path.exists())

if data_path.exists():
    print("File size (MB):", round(data_path.stat().st_size / (1024 * 1024), 2))

File exists: True
File size (MB): 22.62


In [19]:
import pandas as pd
import time

start = time.time()

print("Loading Excel file...")

df = pd.read_excel(data_path, engine="openpyxl")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Time taken:", round(time.time() - start, 2), "seconds")

Loading Excel file...
Dataset loaded successfully!
Rows: 541909
Columns: 8
Time taken: 71.33 seconds


In [20]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)


Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows: 5268

Data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [21]:
print("Cancelled invoices:",
      df["InvoiceNo"].astype(str).str.startswith("C").sum())

print("Zero or negative quantities:",
      (df["Quantity"] <= 0).sum())

print("Zero or negative unit prices:",
      (df["UnitPrice"] <= 0).sum())

print("Unique customers:",
      df["CustomerID"].nunique())

print("Date range:")
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())


Cancelled invoices: 9288
Zero or negative quantities: 10624
Zero or negative unit prices: 2517
Unique customers: 4372
Date range:
2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [22]:
# Keep the original df unchanged
df_clean = df.copy()

# 1. Remove exact duplicate rows
df_clean = df_clean.drop_duplicates().copy()

# 2. Exclude rows without a customer ID
df_clean = df_clean.dropna(subset=["CustomerID"]).copy()

# 3. Exclude cancelled invoices
df_clean = df_clean[
    ~df_clean["InvoiceNo"].astype(str).str.startswith("C")
].copy()

# 4. Keep only positive quantities and prices
df_clean = df_clean[
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

# 5. Convert customer IDs to integer
df_clean["CustomerID"] = df_clean["CustomerID"].astype("int64")

# 6. Calculate line-item revenue
df_clean["Revenue"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print("Unique customers remaining:", df_clean["CustomerID"].nunique())
print("Missing customer IDs:", df_clean["CustomerID"].isna().sum())
print("Remaining duplicates:", df_clean.duplicated().sum())
print("Total revenue:", round(df_clean["Revenue"].sum(), 2))

display(df_clean.head())

Original rows: 541909
Clean rows: 392692
Rows removed: 149217
Unique customers remaining: 4338
Missing customer IDs: 0
Remaining duplicates: 0
Total revenue: 8887208.89


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [23]:
# Set the analysis date one day after the last transaction
analysis_date = df_clean["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

rfm = (
    df_clean.groupby("CustomerID")
    .agg(
        LastPurchaseDate=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

# Calculate recency in days
rfm["Recency"] = (
    analysis_date - rfm["LastPurchaseDate"].dt.normalize()
).dt.days

# Arrange columns
rfm = rfm[
    ["CustomerID", "Recency", "Frequency", "Monetary", "LastPurchaseDate"]
]

print("Analysis date:", analysis_date.date())
print("Number of customers:", len(rfm))
print("\nRFM summary:")
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

print("\nSample customers:")
display(rfm.head(10))

Analysis date: 2011-12-10
Number of customers: 4338

RFM summary:


,Recency,Frequency,Monetary
count,4338.00,4338.00,4338.00
mean,93.06,4.27,2048.69
std,100.01,7.70,8985.23
min,1.00,1.00,3.75
25%,18.00,1.00,306.48
50%,51.00,2.00,668.57
75%,142.75,5.00,1660.60
max,374.00,209.00,280206.02



Sample customers:


,CustomerID,Recency,Frequency,Monetary,LastPurchaseDate
0,12346,326,1,77183.60,2011-01-18 10:01:00
1,12347,3,7,4310.00,2011-12-07 15:52:00
2,12348,76,4,1797.24,2011-09-25 13:13:00
3,12349,19,1,1757.55,2011-11-21 09:51:00
4,12350,311,1,334.40,2011-02-02 16:01:00
5,12352,37,8,2506.04,2011-11-03 14:37:00
6,12353,205,1,89.00,2011-05-19 17:47:00
7,12354,233,1,1079.40,2011-04-21 13:11:00
8,12355,215,1,459.40,2011-05-09 13:49:00
9,12356,23,3,2811.43,2011-11-17 08:40:00


In [24]:
# Recency: fewer days since purchase = higher score
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: more orders = higher score
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary: higher gross sales = higher score
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined RFM score
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

# Numeric total for convenient sorting
rfm["RFM_Total"] = (
    rfm["R_Score"] +
    rfm["F_Score"] +
    rfm["M_Score"]
)

print("RFM scoring completed!")
display(
    rfm[
        ["CustomerID", "Recency", "Frequency", "Monetary",
         "R_Score", "F_Score", "M_Score", "RFM_Score", "RFM_Total"]
    ].head(10)
)

RFM scoring completed!


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
0,12346,326,1,77183.60,1,1,5,115,7
1,12347,3,7,4310.00,5,5,5,555,15
2,12348,76,4,1797.24,2,4,4,244,10
3,12349,19,1,1757.55,4,1,4,414,9
4,12350,311,1,334.40,1,1,2,112,4
5,12352,37,8,2506.04,3,5,5,355,13
6,12353,205,1,89.00,1,1,1,111,3
7,12354,233,1,1079.40,1,1,4,114,6
8,12355,215,1,459.40,1,1,2,112,4
9,12356,23,3,2811.43,4,3,5,435,12


In [25]:
print("Score ranges:")
display(rfm[["R_Score", "F_Score", "M_Score"]].agg(["min", "max"]))

print("\nScore distribution:")
for col in ["R_Score", "F_Score", "M_Score"]:
    print(f"\n{col}")
    print(rfm[col].value_counts().sort_index())

print("\nMissing scores:")
print(rfm[["R_Score", "F_Score", "M_Score"]].isna().sum())

Score ranges:


,R_Score,F_Score,M_Score
min,1,1,1
max,5,5,5



Score distribution:

R_Score
R_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

F_Score
F_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

M_Score
M_Score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

Missing scores:
R_Score    0
F_Score    0
M_Score    0
dtype: int64


In [26]:
def assign_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    else:
        return "Lost"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

segment_summary = (
    rfm.groupby("Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Gross_Sales=("Monetary", "sum"),
        Average_Customer_Sales=("Monetary", "mean")
    )
    .reset_index()
)

segment_summary["Customer_Share_%"] = (
    segment_summary["Customers"] / rfm["CustomerID"].nunique() * 100
).round(2)

segment_summary["Revenue_Share_%"] = (
    segment_summary["Gross_Sales"] / rfm["Monetary"].sum() * 100
).round(2)

display(segment_summary.round(2))

,Segment,Customers,Gross_Sales,Average_Customer_Sales,Customer_Share_%,Revenue_Share_%
0,At Risk,466,766529.58,1644.91,10.74,8.63
1,Champions,941,5741913.58,6101.93,21.69,64.61
2,Lost,1932,885536.99,458.35,44.54,9.96
3,Loyal,999,1493228.74,1494.72,23.03,16.80


In [27]:
segment_check = (
    rfm.groupby("Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Avg_Recency_Days=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_RFM_Score=("RFM_Total", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .round(2)
)

display(segment_check)

,Customers,Avg_Recency_Days,Avg_Frequency,Avg_RFM_Score,Avg_Monetary
Segment,,,,,
At Risk,466,140.18,3.82,9.20,1644.91
Champions,941,12.98,11.21,13.94,6101.93
Lost,1932,151.36,1.27,5.73,458.35
Loyal,999,33.76,3.74,10.57,1494.72


In [28]:
# Check the recency and frequency spread within Lost
lost_check = (
    rfm[rfm["Segment"] == "Lost"]
    .groupby(
        pd.cut(
            rfm.loc[rfm["Segment"] == "Lost", "Recency"],
            bins=[0, 30, 60, 90, 180, 365, float("inf")],
            labels=["0-30", "31-60", "61-90", "91-180", "181-365", "365+"],
            include_lowest=True
        )
    )
    .agg(
        Customers=("CustomerID", "count"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .round(2)
)

display(lost_check)

,Customers,Avg_Frequency,Avg_Monetary
Recency,,,
0-30,278,1.26,439.29
31-60,272,1.17,465.05
61-90,275,1.29,535.08
91-180,356,1.49,415.71
181-365,679,1.23,475.86
365+,72,1.03,259.29


In [29]:
def assign_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    elif r >= 3 and f <= 2:
        return "New / Potential"
    else:
        return "Lost"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

segment_check = (
    rfm.groupby("Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Avg_Recency_Days=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_RFM_Score=("RFM_Total", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .round(2)
    .sort_values("Customers", ascending=False)
)

display(segment_check)

,Customers,Avg_Recency_Days,Avg_Frequency,Avg_RFM_Score,Avg_Monetary
Segment,,,,,
Lost,1269,211.10,1.31,4.99,460.28
Loyal,999,33.76,3.74,10.57,1494.72
Champions,941,12.98,11.21,13.94,6101.93
New / Potential,663,37.02,1.20,7.16,454.66
At Risk,466,140.18,3.82,9.20,1644.91


In [30]:
# Final quality checks
print("Total customers:", rfm["CustomerID"].nunique())
print("Missing segments:", rfm["Segment"].isna().sum())
print("Duplicate customer records:", rfm["CustomerID"].duplicated().sum())

print("\nSegment customer counts:")
print(rfm["Segment"].value_counts())

print("\nRFM score ranges:")
print(rfm[["R_Score", "F_Score", "M_Score"]].agg(["min", "max"]))

Total customers: 4338
Missing segments: 0
Duplicate customer records: 0

Segment customer counts:
Segment
Lost               1269
Loyal               999
Champions           941
New / Potential     663
At Risk             466
Name: count, dtype: int64

RFM score ranges:
     R_Score  F_Score  M_Score
min        1        1        1
max        5        5        5


In [31]:
from pathlib import Path
import pandas as pd

# Output folder
output_dir = Path(
    r"C:\Users\Lenovo\Desktop\Customer_Segmentation_RFM\output"
)
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Final customer-level RFM dataset
rfm.to_csv(
    output_dir / "customer_rfm.csv",
    index=False
)

# 2. Segment-level summary
segment_summary = (
    rfm.groupby("Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Gross_Sales=("Monetary", "sum"),
        Average_Customer_Sales=("Monetary", "mean"),
        Avg_Recency_Days=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean")
    )
    .reset_index()
)

segment_summary["Customer_Share_%"] = (
    segment_summary["Customers"] / rfm["CustomerID"].nunique() * 100
)

segment_summary["Gross_Sales_Share_%"] = (
    segment_summary["Gross_Sales"] / rfm["Monetary"].sum() * 100
)

segment_summary = segment_summary.round(2)

segment_summary.to_csv(
    output_dir / "segment_summary.csv",
    index=False
)

# 3. Clean transaction-level data (optional, for deeper analysis)
df_clean.to_csv(
    output_dir / "cleaned_transactions.csv",
    index=False
)

# Verify exported files
for file in [
    "customer_rfm.csv",
    "segment_summary.csv",
    "cleaned_transactions.csv"
]:
    path = output_dir / file
    print(f"{file}: {path.stat().st_size / 1024:.1f} KB")

print("\nCustomer records:", len(rfm))
print("Segment records:", len(segment_summary))
print("Export complete.")

customer_rfm.csv: 261.1 KB
segment_summary.csv: 0.4 KB
cleaned_transactions.csv: 37162.5 KB

Customer records: 4338
Segment records: 5
Export complete.
